# This is the Notebook that does the Preamble Work for all Out-of-Core Models

The feature processing pipeline is same for all Models and thus I created a notebook to organize all of that work into a single place for conveniecne.

**The Notebook is part of the US Accidents Analysis Project**

**Link - https://www.kaggle.com/work/collections/18305074**

*Please Note, the Warnings have been specifically left on; so that the reader is aware of any poential changes or behaviour specifications.*

*Class Weights & Sample Weights calculation has specifically left to each model's notebook to ensure index allignment*

In [1]:
!git clone --filter=blob:none --no-checkout "https://github.com/Paras-GaurLRN/US-Accidents-EDA-Ensemble-Models.git"
%cd "US-Accidents-EDA-Ensemble-Models"
!git sparse-checkout init --cone
!git sparse-checkout set "notebooks/US Accidents - Pipelines/"
!git checkout main
%cd ..

Cloning into 'US-Accidents-EDA-Ensemble-Models'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 112 (delta 61), reused 77 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 13.60 KiB | 4.53 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/kaggle/working/US-Accidents-EDA-Ensemble-Models
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 1), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 17.92 MiB | 21.72 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (12/12), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
/kaggle/working


In [2]:
print("Files In The Repo-\n")

data_path = '/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines'

import os
for _, _, filenames in os.walk(data_path):
    for filename in filenames:
        print(filename)

Files In The Repo-

Imports.txt
Libraries.txt
pipeline.py
us-accidents-pipeline.ipynb


In [3]:
print("The Dataset-\n")

dataset_path = '/kaggle/input'

for _, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(filename)

The Dataset-

US_Accidents_March23.csv
__results__.html
__notebook__.ipynb
__output__.json
custom.css


In [4]:
import sys

sys.path.append(data_path)

# To ensure that we can import the transformers

In [5]:
print("### Libraires ###\n")
with open(f'{data_path}/Libraries.txt') as LibrariesTXT:
    for line in LibrariesTXT.readlines():
        print(line)

### Libraires ###

scikit-learn

imbalanced-learn

feature-engine


In [6]:
print("### Imports ###\n")
with open(f'{data_path}/Imports.txt') as ImportsTXT:
    for line in ImportsTXT.readlines():
        print(line)

### Imports ###

from warnings import warn

from sklearn.base import (BaseEstimator, TransformerMixin, clone)

from sklearn.utils._param_validation import StrOptions

from imblearn.base import BaseSampler

from sklearn.utils.validation import check_is_fitted

from sklearn.compose import ColumnTransformer

from feature_engine.datetime import DatetimeFeatures

from feature_engine.outliers import ArbitraryOutlierCapper

from imblearn.pipeline import Pipeline as IMBPipe

import pandas as pd

import numpy as np


In [7]:
!pip install feature-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 4.7 MB/s eta 0:00:00


In [8]:
!pip list | grep -E "numpy|pandas|feature-engine|scikit-learn"

geopandas                                1.1.3
numpy                                    2.0.2
pandas                                   2.3.3
pandas-datareader                        0.10.0
pandas-gbq                               0.30.0
pandas-profiling                         3.6.6
pandas-stubs                             2.2.2.240909
pandasql                                 0.7.3
scikit-learn                             1.6.1
sklearn-pandas                           2.2.0


**Note: Target column = Severity**

In [9]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import (StandardScaler, OrdinalEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [10]:
from pipeline import (AnomalyCleaner, DateTimeFeatureEngineer, ColumnDropper, Illuminator, OutOfCoreNumericalImputer)

**We Need To Discuss all the decisions made in the following code here,**

**1)** Firstly, the loop is made accordingly due to different transformers behaving differently.

We have 3 clasifications-

*SL* : Stateless = These do not store anything that would need to be propogated across the loop.

*SI* : Stateful, but Incremental = These require propogation of learnt values across fits, but are build for Incremental Learning.

*SN* : Stateful, non Incremental = These are Stateful but not built for Incremental Learning.

*SL* & *SI* are easy to work with, but *SN* require inspection in each loop. We also need to ensure no **Evaluation Leakage** whatsoever.

**2)** The First iteration is the learning and logging stage; it requires the most attention, rest stages are simple.

**3)** All iterations forward need to account for the State of a transformer, past values & appropriate transformations.

**The Column Map**

*Pre-categories encoded for simplicity, OHE makes the final result a bit larger*
![Column Map](https://i.postimg.cc/zvyyYypX/Model-1.png)

In [11]:
from sklearn import set_config
set_config(transform_output='pandas')

# Train-Test Spliting The Data

*Chunk Wise Data Splitting Due to Size*

In [12]:
from sklearn.model_selection import train_test_split

FILE = '/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv'

TRAIN_FILE = 'US_Accidents_train.csv'
TEST_FILE = 'US_Accidents_test.csv'

first_train = True
first_test = True

for chunk in pd.read_csv(
    FILE,
    index_col='ID',
    chunksize=20_000
):
    train_chunk, test_chunk = train_test_split(
        chunk,
        test_size=0.30,
        random_state=34
    )

    train_chunk.to_csv(
        TRAIN_FILE,
        mode='w' if first_train else 'a',
        header=first_train,
        index=True
    )

    test_chunk.to_csv(
        TEST_FILE,
        mode='w' if first_test else 'a',
        header=first_test,
        index=True
    )

    first_train = False
    first_test = False

    del train_chunk, test_chunk, chunk

print("Data Split Created!")

Data Split Created!


# Transformers Training

*The Feature Processing is same for all Out-of-Core Models, Hence this can be done seperately*

Note: The State-ness of the Transformers cannot be handled by imblearn.pipeline.Pipeline so we need to ensure proper training by hand.

Transformers have been marked with their state-type and we can thus infer why the forecoming training setup is the way it is

In [13]:
drop_cols = ["Amenity","Bump","Crossing","Give_Way","Junction","No_Exit","Railway","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal"]
rem_num_cols = ["Temperature(F)","Humidity(%)","Pressure(in)","Visibility(mi)","Wind_Speed(mph)","Precipitation(in)"]
rem_cat_cols = ["Source","Wind_Direction","Weather_Condition","Illumination"]
rem_datetime_cols = ["Accident Day","Accident Timing","Accident Year","Weekend"]
first_iter = True

T1_AC = AnomalyCleaner() # SL
T2_DTFE = DateTimeFeatureEngineer() # SL
T3_IL = Illuminator() # SL
T4_CD = ColumnDropper(columns= drop_cols + [col for col in ColumnDropper.DEFAULT_COLUMNS if col != 'Precipitation(in)']) # SL
T5_SIM_CAT = SimpleImputer(strategy='constant',fill_value='Missing',copy=False) # SL, categorical imputer is SL but a numerical analog will be SN
T5_SIM_NUM = OutOfCoreNumericalImputer(file_name = '/kaggle/working/US_Accidents_train.csv',
                                       columns = rem_num_cols) # SL

T5_SIM = ColumnTransformer([
    ('num',T5_SIM_NUM,rem_num_cols),
    ('cat',T5_SIM_CAT,[c for c in rem_cat_cols if c != 'Source'])
],remainder='passthrough',n_jobs=-1,verbose=False,verbose_feature_names_out=False) # SL, due to the Transformers used

T6_ENC_OE = OrdinalEncoder(categories=[['Missing',*(val for key,val in T3_IL.DEFAULT_ILLUMINATION_ORDER.items() if key != 'Miscellaneous')]],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1,
                           encoded_missing_value=-1) # SI

T6_ENC_OHE = OneHotEncoder(categories='auto',drop='first',sparse_output=False,handle_unknown='infrequent_if_exist',min_frequency=1e-5) # SI

T6_ENC = ColumnTransformer([
    ('ohe',T6_ENC_OHE,['Wind_Direction','Weather_Condition']),
    ('ord',T6_ENC_OE,['Illumination'])
],remainder='passthrough',n_jobs=-1,verbose=False,verbose_feature_names_out=False) # SI, due to the transformers used

T7_SS = StandardScaler() # SI

for chunk in pd.read_csv('/kaggle/working/US_Accidents_train.csv',
                         index_col='ID',
                         chunksize=20000):
    if first_iter:
        X, y = chunk.drop(columns=['Severity']), chunk['Severity']
        
        Wind_Direction = (
            pd.read_csv(
                '/kaggle/working/US_Accidents_train.csv',
                usecols=['Wind_Direction']
            )['Wind_Direction']
        )

        Weather_Condition = (
            pd.read_csv(
                '/kaggle/working/US_Accidents_train.csv',
                usecols=['Weather_Condition']
            )['Weather_Condition']
        )
        
        T1_AC.fit(X)
        X, y = T1_AC.fit_resample(X, y)
        
        X = T2_DTFE.fit_transform(X)
        
        X = T3_IL.fit_transform(X)
        
        X = T4_CD.fit_transform(X)
        
        X = T5_SIM.fit_transform(X)
        
        X = T6_ENC.fit(pd.DataFrame({
            'Wind_Direction' : Wind_Direction.fillna('Missing'),
            'Weather_Condition' : Weather_Condition.fillna('Missing'),
            'Illumination' : np.full(shape=Wind_Direction.to_numpy().shape,fill_value='I')
        })).transform(X)

        del Weather_Condition
        del Wind_Direction
        
        T7_SS.partial_fit(X)
        
        first_iter = False
    else:
        X, y = chunk.drop(columns=['Severity']), chunk['Severity']

        X, y = T1_AC.fit_resample(X, y)
        X = T2_DTFE.transform(X)
        X = T3_IL.transform(X)
        X = T4_CD.transform(X)
        X = T5_SIM.transform(X)
        X = T6_ENC.transform(X)
        T7_SS.partial_fit(X)

print("Transformers Trained!")

/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines/pipeline.py:97: UserWarning: AnomalyCleaner has a dual sampler/transformer interface and is customized for compatibility with imbalanced-learn's Pipeline API. When used inside an imblearn Pipeline, AnomalyCleaner is treated as a sampler: it participates during fitting via fit_resample(), but samplers are skipped during prediction/inference. Consequently, transform()-based capping of anomalous observations will not be applied automatically by pipeline.predict(). To apply inference-time capping, either use AnomalyCleaner.transform() independently before passing the data to the remaining pipeline steps, or treat validation and capping of prediction data as the responsibility of the calling API.
  warn(


Transformers Trained!


# Sinking Learnt Transformers

In [14]:
import joblib

joblib.dump(
    {
        'T1_AC' : T1_AC,
        'T2_DTFE' : T2_DTFE,
        'T3_IL' : T3_IL,
        'T4_CD' : T4_CD,
        'T5_SIM' : T5_SIM,
        'T6_ENC' : T6_ENC,
        'T7_SS' : T7_SS
    },
    'Transformers.pkl'
)

print("Transformers Sinked!")

Transformers Sinked!
